In [217]:
import pandas as pd

recipes_df = pd.read_csv("./data/recipes.csv")

def gettransitionModel(recipes_df):
    result = {}

    for _, row in recipes_df.iterrows():
        meal_type = row["type"]
        meal_name = row["Name"]

        meal_info = (
            row["Category"],
            row["total_price"],
            row["provided_calories"],
            row["provided_protein"],
            row["provided_carbs"],
            row["provided_fat"]
        )

        if meal_type not in result:
            result[meal_type] = {}

        result[meal_type][meal_name] = meal_info

    return result

transition_model = gettransitionModel(recipes_df)

In [218]:
NUTRI_PREFERENCE_MAP = {
    "maintain":       {"protein": 0.30, "fat": 0.25, "carbs": 0.45},
    "gain":   {"protein": 0.40, "fat": 0.25, "carbs": 0.35},
    "loss":       {"protein": 0.25, "fat": 0.35, "carbs": 0.40},
}

In [219]:
class MealPlannerState:
    def __init__(self, day_number, meal_type, meal, remaining_budget, today_calorie_use, today_protein_use, today_fat_use, today_carb_use, used_meals = None):
        self.day = day_number
        self.meal_type = meal_type
        self.meal = meal
        self.remaining_budget = remaining_budget
        self.today_calorie_use = today_calorie_use
        self.today_protein_use = today_protein_use
        self.today_fat_use = today_fat_use
        self.today_carb_use = today_carb_use
        self.used_meals = used_meals
        
    def __eq__(self, other):
        return isinstance(other, MealPlannerState) and \
            self.day == other.day and \
            self.meal_type == other.meal_type and \
            self.meal == other.meal

    def __hash__(self):
        return hash((self.day, self.meal_type, self.meal))
        

In [220]:
class Node:    
    def __init__(self, state, parent=None, action=None, cost=0, heuristic=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = (parent.g + cost) if parent else 0
        self.f = self.g + heuristic
        self.depth = 0 if parent is None else parent.depth + 1

    def path(self):
        node = self
        actions = []
        while node.parent is not None:
            actions.append(node.action)
            node = node.parent
        actions.reverse()
        return actions

    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):
        return hash(self.state)


In [221]:
class MealPlanningProblem:

    def __init__(self, transition_model, TDEE, total_budget, num_days, macroNutri_preference = "maintain"):
        self.transition_model = transition_model
        self.total_budget = total_budget
        self.TDEE = TDEE
        self.num_days = num_days
        self.total_slots = num_days * 3
        self.meal_types = ['Breakfast', 'Lunch', 'Dinner']

        self.meal_type_weights = {
            'Breakfast': 0.25,
            'Lunch': 0.40,
            'Dinner': 0.35,
        }

        self.macro_nutrients_ratios = NUTRI_PREFERENCE_MAP[macroNutri_preference]

        # Create initial state: starting with Breakfast (meal_type=0)
        self.initial_state = MealPlannerState(
            day_number=0,
            meal_type=0,
            meal=None,
            remaining_budget=total_budget,
            today_calorie_use=0,
            today_protein_use=0,
            today_fat_use=0,
            today_carb_use=0,
            used_meals=dict()
        )

    def _calculate_totals(self, recipes_list):
        total_cost = 0
        total_calories = 0
        
        i = 0
        for meal_name in recipes_list:
            recipe = self.transition_model[self.meal_types[i]][meal_name]
            total_cost += recipe[1]
            total_calories += recipe[2]

            i = (i+1)%3

        return total_cost, total_calories

    def is_goal(self, node):
        state = node.state
        
        if state.day < self.num_days:
            return False
        
        recipes_chosen = node.path()
        
        if len(recipes_chosen) != self.total_slots:
            return False
        
        total_cost, total_calories = self._calculate_totals(recipes_chosen)
        
        goal_calories = self.TDEE * self.num_days
        calorie_tolerance = goal_calories * 0.1
        calories_ok = abs(total_calories - goal_calories) <= calorie_tolerance
        
        budget_ok = total_cost <= self.total_budget
        
        return calories_ok and budget_ok

    def expand_node(self, node, use_cost=True, use_heuristic=False):
        state = node.state
        
        # Check if we've filled all meal slots
        current_slot = node.depth
        if current_slot >= self.total_slots:
            return []
        
        children = []
        current_meal_type_idx = current_slot % 3
        current_meal_type = self.meal_types[current_meal_type_idx]
        
        # Get available meals for this meal type
        valid_actions = self.transition_model[current_meal_type]
        
        for meal_name in valid_actions:
            # Skip if meal already used
            if state.used_meals and meal_name in state.used_meals.keys() and state.used_meals[meal_name] >= 1:
                continue

            # Get meal info from transition model: (category, total_price, calories, protein, carbs, fat)
            meal_info = self.transition_model[current_meal_type][meal_name]
            total_price = meal_info[1]
            provided_calories = meal_info[2]
            provided_protein = meal_info[3]
            provided_carbs = meal_info[4]
            provided_fat = meal_info[5]
            
            action_cost = self.calculate_cost(current_slot, current_meal_type, meal_name) if use_cost else 0
            
            # Calculate new state values
            new_meal_type_idx = (current_meal_type_idx + 1) % 3
            new_day = state.day if new_meal_type_idx != 0 else state.day + 1
            new_remaining_budget = state.remaining_budget - total_price
            new_calorie_use = 0 if new_meal_type_idx == 0 else state.today_calorie_use + provided_calories
            new_protein_use = 0 if new_meal_type_idx == 0 else state.today_protein_use + provided_protein
            new_carb_use = 0 if new_meal_type_idx == 0 else state.today_carb_use + provided_carbs
            new_fat_use = 0 if new_meal_type_idx == 0 else state.today_fat_use + provided_fat
            
            # Track used meals
            new_used_meals = dict(state.used_meals)  if state.used_meals and new_day % 15 != 0 else dict({meal_name: 1})
            new_used_meals[meal_name] = new_used_meals[meal_name] + 1 if meal_name in new_used_meals.keys() else 1
            
            # Create new state
            new_state = MealPlannerState(
                day_number=new_day,
                meal_type=new_meal_type_idx,
                meal=meal_name,
                remaining_budget=new_remaining_budget,
                today_calorie_use=new_calorie_use,
                today_protein_use=new_protein_use,
                today_fat_use=new_fat_use,
                today_carb_use=new_carb_use,
                used_meals=new_used_meals
            )
            
            heuristic = self.calculate_heuristic(new_state) if use_heuristic else 0
            
            # Create child node
            child = Node(state=new_state, parent=node, action=meal_name, cost=action_cost, heuristic=heuristic)
            children.append(child)

        
        return children

    def calculate_cost(self, current_slot, meal_type, meal_name):
        # Get meal info from transition model
        meal_info = self.transition_model[meal_type][meal_name]
        _, meal_price, meal_calories, meal_protein, meal_carb, meal_fat = meal_info
        
        # Calculate remaining slots and days
        remaining_slots = max(1, self.total_slots - current_slot)
        remaining_days = max(1, self.num_days - (current_slot // 3))
        
        # Calculate allocated budget and nutrients for this meal type
        allocated_price = self.meal_type_weights[meal_type] * (self.total_budget / remaining_slots)

        allocated_calories = self.meal_type_weights[meal_type] * self.TDEE

        allocated_protein = self.macro_nutrients_ratios["protein"] * allocated_calories
        allocated_carb = self.macro_nutrients_ratios["carbs"] * allocated_calories
        allocated_fat = self.macro_nutrients_ratios["fat"] * allocated_calories
        
        # Calculate deviations
        price_deviation = abs(meal_price - allocated_price)

        calorie_deviation = abs(meal_calories - allocated_calories)
        
        protein_deviation = abs(meal_protein - allocated_protein)
        carb_deviation = abs(meal_carb - allocated_carb)
        fat_deviation = abs(meal_fat - allocated_fat)

        # Normalize deviations
        normalized_price_dev = price_deviation / allocated_price if allocated_price > 0 else 0

        normalized_cal_dev = calorie_deviation / allocated_calories if allocated_calories > 0 else 0
        
        normalized_protein_dev = protein_deviation / allocated_protein if allocated_protein > 0 else 0
        normalized_carb_dev = carb_deviation / allocated_carb if allocated_carb > 0 else 0
        normalized_fat_dev = fat_deviation / allocated_fat if allocated_fat > 0 else 0

        total_cost = (normalized_price_dev + normalized_cal_dev + normalized_protein_dev + normalized_carb_dev + normalized_fat_dev) / 5.0

        return total_cost

    
    def calculate_heuristic(self, state):

        protein_w = self.macro_nutrients_ratios["protein"]
        carbs_w = self.macro_nutrients_ratios["carbs"]
        fat_w = self.macro_nutrients_ratios["fat"]
        
        target_protein = protein_w * self.TDEE
        target_carbs   = carbs_w * self.TDEE
        target_fat     = fat_w * self.TDEE
    
        nutrition_error = (
            abs(state.today_calorie_use -  self.TDEE ) +
            abs(state.today_protein_use - target_protein) +
            abs(state.today_carb_use - target_carbs) +
            abs(state.today_fat_use - target_fat)
        )
    
        # budget_error = max(0, - state.remaining_budget)
    
        # expected_ratio = self.meal_type_weights[self.meal_types[state.meal_type]]
    
        # expected_cal = expected_ratio * self.TDEE
    
        # meal_error = abs(state.today_calorie_use - expected_cal)
    
        # meal_penalty = max(0, meal_error - 0.1 * expected_cal)
    
        return nutrition_error

In [222]:
import queue

class AstarSearch:
    def __init__(self,problem):
        self.problem = problem
        self.frontier = queue.PriorityQueue()

        self.explored = set()

    def search(self):

        node = Node(self.problem.initial_state)
        self.frontier.put(node)
        
        while True:
            if self.frontier.empty():
                return None

            node = self.frontier.get()
            
            if self.problem.is_goal(node):
                solution = self._get_solution_path(node)
                return solution

            self.explored.add(node.state)

            children = self.problem.expand_node(node, True, True)

            for child in children:
                if child.state not in self.explored and child not in self.frontier.queue:
                    self.frontier.put(child)

    def _get_solution_path(self, solution_node):
        path = solution_node.path()
        path = [tuple([
                path[i],
                path[i+1],
                path[i+2]
            ]) for i in range(0, len(path), 3)]

        return path



In [223]:
import queue

class GreedySearch:
    def __init__(self,problem):
        self.problem = problem
        self.frontier = queue.PriorityQueue()

        self.explored = set()

    def search(self):

        node = Node(self.problem.initial_state)
        self.frontier.put(node)
        
        while True:
            if self.frontier.empty():
                return None

            node = self.frontier.get()
            
            if self.problem.is_goal(node):
                solution = self._get_solution_path(node)
                return solution

            self.explored.add(node.state)

            children = self.problem.expand_node(node, False, True)

            for child in children:
                if child.state not in self.explored and child not in self.frontier.queue:
                    self.frontier.put(child)

    def _get_solution_path(self, solution_node):
        path = solution_node.path()
        path = [tuple([
                path[i],
                path[i+1],
                path[i+2]
            ]) for i in range(0, len(path), 3)]

        return path



In [224]:
def test_a_star(TDEE, budget, days):
    problem = MealPlanningProblem(transition_model, TDEE, budget, days)
    A_star = AstarSearch(problem)

    solution = A_star.search()

    if solution is None:
        print("couldn't find a suitable plan")
        return

    # Helper function to find meal info by name
    def find_meal(meal_name):
        for meal_type in transition_model:
            if meal_name in transition_model[meal_type]:
                return transition_model[meal_type][meal_name]
        return None
    
    # Extract recipes from solution and calculate statistics
    recipe_data = []
    for day_meals in solution:
        for meal_name in day_meals:
            meal_info = find_meal(meal_name)
            if meal_info:
                recipe_data.append(meal_info)
    
    # Calculate totals (note: protein, carbs, fat are stored as calorie contributions)
    total_cost = sum(r[1] for r in recipe_data)  # total_price at index 1
    total_calories = sum(r[2] for r in recipe_data)  # provided_calories at index 2
    total_protein_cal = sum(r[3] for r in recipe_data)  # provided_protein (in calories) at index 3
    total_carbs_cal = sum(r[4] for r in recipe_data)  # provided_carbs (in calories) at index 4
    total_fat_cal = sum(r[5] for r in recipe_data)  # provided_fat (in calories) at index 5
    
    # Convert calorie contributions to grams
    total_protein_g = total_protein_cal / 4
    total_carbs_g = total_carbs_cal / 4
    total_fat_g = total_fat_cal / 9
    
    # Calculate macronutrient ratios as percentages of total calories
    actual_protein_ratio = (total_protein_cal / total_calories * 100) if total_calories > 0 else 0
    actual_carbs_ratio = (total_carbs_cal / total_calories * 100) if total_calories > 0 else 0
    actual_fat_ratio = (total_fat_cal / total_calories * 100) if total_calories > 0 else 0
    
    # Get target macronutrient ratios
    target_ratios = NUTRI_PREFERENCE_MAP.get("maintain", {"protein": 0.30, "fat": 0.25, "carbs": 0.45})
    target_protein_pct = target_ratios["protein"] * 100
    target_carbs_pct = target_ratios["carbs"] * 100
    target_fat_pct = target_ratios["fat"] * 100
    
    # Calculate deviations
    expected_calories = TDEE * days
    calorie_deviation = total_calories - expected_calories
    calorie_deviation_pct = (calorie_deviation / expected_calories) * 100 if expected_calories > 0 else 0
    
    budget_remaining = budget - total_cost
    budget_deviation_pct = (budget_remaining / budget) * 100 if budget > 0 else 0
    
    # Macronutrient deviations
    protein_dev = actual_protein_ratio - target_protein_pct
    carbs_dev = actual_carbs_ratio - target_carbs_pct
    fat_dev = actual_fat_ratio - target_fat_pct
    
    # Daily breakdown
    daily_costs = []
    daily_calories = []
    daily_protein = []
    daily_carbs = []
    daily_fat = []
    for i, day_meals in enumerate(solution):
        day_cost = 0
        day_calories = 0
        day_protein_cal = 0
        day_carbs_cal = 0
        day_fat_cal = 0
        for meal_name in day_meals:
            meal_info = find_meal(meal_name)
            if meal_info:
                day_cost += meal_info[1]  # total_price
                day_calories += meal_info[2]  # provided_calories
                day_protein_cal += meal_info[3]  # provided_protein (calories)
                day_carbs_cal += meal_info[4]  # provided_carbs (calories)
                day_fat_cal += meal_info[5]  # provided_fat (calories)
        daily_costs.append(day_cost)
        daily_calories.append(day_calories)
        daily_protein.append(day_protein_cal) 
        daily_carbs.append(day_carbs_cal) 
        daily_fat.append(day_fat_cal)  
    
    # Unique meals
    unique_meals = set(meal_name for day_meals in solution for meal_name in day_meals)
    
    # Print results
    print("="*60)
    print("A* MEAL PLAN RESULTS")
    print("="*60)
    print(f"\n PARAMETERS:")
    print(f"  Daily TDEE target: {TDEE} cal")
    print(f"  Total budget: {budget} DZD")
    print(f"  Duration: {days} days")
    
    print(f"\n COST ANALYSIS:")
    print(f"  Total cost: {total_cost:.2f} DZD")
    print(f"  Budget remaining: {budget_remaining:.2f} DZD ({budget_deviation_pct:.1f}%)")
    print(f"  Average daily cost: {total_cost/days:.2f} DZD")
    
    print(f"\n CALORIE ANALYSIS:")
    print(f"  Total calories: {total_calories:.0f} cal")
    print(f"  Expected calories: {expected_calories} cal")
    print(f"  Deviation: {calorie_deviation:+.0f} cal ({calorie_deviation_pct:+.1f}%)")
    print(f"  Average daily calories: {total_calories/days:.0f} cal")
    
    print(f"\n MACRONUTRIENT DISTRIBUTION:")
    print(f"  Protein: {total_protein_g:.0f} ({actual_protein_ratio:.1f}%) [Target: {target_protein_pct:.0f}%] {protein_dev:+.1f}%")
    print(f"  Carbs:   {total_carbs_g:.0f} ({actual_carbs_ratio:.1f}%) [Target: {target_carbs_pct:.0f}%] {carbs_dev:+.1f}%")
    print(f"  Fat:     {total_fat_g:.0f} ({actual_fat_ratio:.1f}%) [Target: {target_fat_pct:.0f}%] {fat_dev:+.1f}%")
    print(f"\n  Daily Avg Macros:")
    print(f"    Protein: {total_protein_g/days:.0f}g | Carbs: {total_carbs_g/days:.0f}g | Fat: {total_fat_g/days:.0f}g")
    
    print(f"\n DAILY BREAKDOWN:")
    for day in range(days):
        status = "✓" if abs(daily_calories[day] - TDEE) / TDEE < 0.1 else "⚠"
        day_protein_ratio = ((daily_protein[day] / daily_calories[day]) * 100) if daily_calories[day] > 0 else 0
        day_carbs_ratio = ((daily_carbs[day] / daily_calories[day]) * 100) if daily_calories[day] > 0 else 0
        day_fat_ratio = ((daily_fat[day] / daily_calories[day]) * 100) if daily_calories[day] > 0 else 0
        print(f"  Day {day+1}: {daily_costs[day]:.2f} DZD, {daily_calories[day]:.0f} cal {status}")
        print(f"           P:{daily_protein[day]:.0f}g({day_protein_ratio:.0f}%) | C:{daily_carbs[day]:.0f}g({day_carbs_ratio:.0f}%) | F:{daily_fat[day]:.0f}g({day_fat_ratio:.0f}%)")
    
    print(f"\n MEAL DIVERSITY:")
    print(f"  Unique meals: {len(unique_meals)} out of {len(solution) * 3}")
    
    print(f"\nMEAL PLAN:")
    for day in range(days):
        print(f"  Day {day+1}: {solution[day]}")
    print("="*60)



In [225]:
##### TESTING #####
test_a_star(2200, 20000, 7)

couldn't find a suitable plan


In [226]:
def test_greedy(TDEE, budget, days):
    problem = MealPlanningProblem(transition_model, TDEE, budget, days)
    A_star = AstarSearch(problem)

    solution = A_star.search()

    if solution is None:
        print("couldn't find a suitable plan")
        return

    # Helper function to find meal info by name
    def find_meal(meal_name):
        for meal_type in transition_model:
            if meal_name in transition_model[meal_type]:
                return transition_model[meal_type][meal_name]
        return None
    
    # Extract recipes from solution and calculate statistics
    recipe_data = []
    for day_meals in solution:
        for meal_name in day_meals:
            meal_info = find_meal(meal_name)
            if meal_info:
                recipe_data.append(meal_info)
    
    # Calculate totals (note: protein, carbs, fat are stored as calorie contributions)
    total_cost = sum(r[1] for r in recipe_data)  # total_price at index 1
    total_calories = sum(r[2] for r in recipe_data)  # provided_calories at index 2
    total_protein_cal = sum(r[3] for r in recipe_data)  # provided_protein (in calories) at index 3
    total_carbs_cal = sum(r[4] for r in recipe_data)  # provided_carbs (in calories) at index 4
    total_fat_cal = sum(r[5] for r in recipe_data)  # provided_fat (in calories) at index 5
    
    # Convert calorie contributions to grams
    total_protein_g = total_protein_cal / 4
    total_carbs_g = total_carbs_cal / 4
    total_fat_g = total_fat_cal / 9
    
    # Calculate macronutrient ratios as percentages of total calories
    actual_protein_ratio = (total_protein_cal / total_calories * 100) if total_calories > 0 else 0
    actual_carbs_ratio = (total_carbs_cal / total_calories * 100) if total_calories > 0 else 0
    actual_fat_ratio = (total_fat_cal / total_calories * 100) if total_calories > 0 else 0
    
    # Get target macronutrient ratios
    target_ratios = NUTRI_PREFERENCE_MAP.get("maintain", {"protein": 0.30, "fat": 0.25, "carbs": 0.45})
    target_protein_pct = target_ratios["protein"] * 100
    target_carbs_pct = target_ratios["carbs"] * 100
    target_fat_pct = target_ratios["fat"] * 100
    
    # Calculate deviations
    expected_calories = TDEE * days
    calorie_deviation = total_calories - expected_calories
    calorie_deviation_pct = (calorie_deviation / expected_calories) * 100 if expected_calories > 0 else 0
    
    budget_remaining = budget - total_cost
    budget_deviation_pct = (budget_remaining / budget) * 100 if budget > 0 else 0
    
    # Macronutrient deviations
    protein_dev = actual_protein_ratio - target_protein_pct
    carbs_dev = actual_carbs_ratio - target_carbs_pct
    fat_dev = actual_fat_ratio - target_fat_pct
    
    # Daily breakdown
    daily_costs = []
    daily_calories = []
    daily_protein = []
    daily_carbs = []
    daily_fat = []
    for i, day_meals in enumerate(solution):
        day_cost = 0
        day_calories = 0
        day_protein_cal = 0
        day_carbs_cal = 0
        day_fat_cal = 0
        for meal_name in day_meals:
            meal_info = find_meal(meal_name)
            if meal_info:
                day_cost += meal_info[1]  # total_price
                day_calories += meal_info[2]  # provided_calories
                day_protein_cal += meal_info[3]  # provided_protein (calories)
                day_carbs_cal += meal_info[4]  # provided_carbs (calories)
                day_fat_cal += meal_info[5]  # provided_fat (calories)
        daily_costs.append(day_cost)
        daily_calories.append(day_calories)
        daily_protein.append(day_protein_cal) 
        daily_carbs.append(day_carbs_cal) 
        daily_fat.append(day_fat_cal)  
    
    # Unique meals
    unique_meals = set(meal_name for day_meals in solution for meal_name in day_meals)
    
    # Print results
    print("="*60)
    print("A* MEAL PLAN RESULTS")
    print("="*60)
    print(f"\n PARAMETERS:")
    print(f"  Daily TDEE target: {TDEE} cal")
    print(f"  Total budget: {budget} DZD")
    print(f"  Duration: {days} days")
    
    print(f"\n COST ANALYSIS:")
    print(f"  Total cost: {total_cost:.2f} DZD")
    print(f"  Budget remaining: {budget_remaining:.2f} DZD ({budget_deviation_pct:.1f}%)")
    print(f"  Average daily cost: {total_cost/days:.2f} DZD")
    
    print(f"\n CALORIE ANALYSIS:")
    print(f"  Total calories: {total_calories:.0f} cal")
    print(f"  Expected calories: {expected_calories} cal")
    print(f"  Deviation: {calorie_deviation:+.0f} cal ({calorie_deviation_pct:+.1f}%)")
    print(f"  Average daily calories: {total_calories/days:.0f} cal")
    
    print(f"\n MACRONUTRIENT DISTRIBUTION:")
    print(f"  Protein: {total_protein_g:.0f} ({actual_protein_ratio:.1f}%) [Target: {target_protein_pct:.0f}%] {protein_dev:+.1f}%")
    print(f"  Carbs:   {total_carbs_g:.0f} ({actual_carbs_ratio:.1f}%) [Target: {target_carbs_pct:.0f}%] {carbs_dev:+.1f}%")
    print(f"  Fat:     {total_fat_g:.0f} ({actual_fat_ratio:.1f}%) [Target: {target_fat_pct:.0f}%] {fat_dev:+.1f}%")
    print(f"\n  Daily Avg Macros:")
    print(f"    Protein: {total_protein_g/days:.0f}g | Carbs: {total_carbs_g/days:.0f}g | Fat: {total_fat_g/days:.0f}g")
    
    print(f"\n DAILY BREAKDOWN:")
    for day in range(days):
        status = "✓" if abs(daily_calories[day] - TDEE) / TDEE < 0.1 else "⚠"
        day_protein_ratio = ((daily_protein[day] / daily_calories[day]) * 100) if daily_calories[day] > 0 else 0
        day_carbs_ratio = ((daily_carbs[day] / daily_calories[day]) * 100) if daily_calories[day] > 0 else 0
        day_fat_ratio = ((daily_fat[day] / daily_calories[day]) * 100) if daily_calories[day] > 0 else 0
        print(f"  Day {day+1}: {daily_costs[day]:.2f} DZD, {daily_calories[day]:.0f} cal {status}")
        print(f"           P:{daily_protein[day]:.0f}g({day_protein_ratio:.0f}%) | C:{daily_carbs[day]:.0f}g({day_carbs_ratio:.0f}%) | F:{daily_fat[day]:.0f}g({day_fat_ratio:.0f}%)")
    
    print(f"\n MEAL DIVERSITY:")
    print(f"  Unique meals: {len(unique_meals)} out of {len(solution) * 3}")
    
    print(f"\nMEAL PLAN:")
    for day in range(days):
        print(f"  Day {day+1}: {solution[day]}")
    print("="*60)



In [227]:
# test_greedy(2200, 300000, 30)